### Notebook for full clean rtsd dataset and mapping to data of Belarus

In [8]:
import os
import json
import shutil
import pandas as pd
from pathlib import Path
from tqdm import tqdm

To check script first check on `train_anno_reduce.json`

In [9]:
# True - `train_anno_reduce.json`
# False - `train_anno.json`

DEBUG = False

Directs path

In [10]:
current_dir = Path(".").resolve()
BASE_DIR = current_dir

for parent in [current_dir] + list(current_dir.parents):
    if (parent / "by_classes_v1.csv").exists() or (parent / "data").exists() or (parent / "train_anno_reduced.json").exists():
        BASE_DIR = parent
        break

if (BASE_DIR / "by_classes_v1.csv").exists():
    CSV_BY_CLASSES = BASE_DIR / "by_classes_v1.csv"
else:
    CSV_BY_CLASSES = BASE_DIR / "docs" / "RTSD_vs_Belarus_Comparison.csv"

if (BASE_DIR / "train_anno_reduced.json").exists():
    DATA_DIR = BASE_DIR
else:
    DATA_DIR = BASE_DIR / "data"

ANNO_TRAIN_FULL = DATA_DIR / "train_anno.json"
ANNO_TRAIN_DEBUG = DATA_DIR / "train_anno_reduced.json"
ANNO_VAL = DATA_DIR / "val_anno.json"

if (BASE_DIR / "rtsd-frames").exists():
    RTSD_FRAMES_DIR = BASE_DIR / "rtsd-frames"
else:
    RTSD_FRAMES_DIR = DATA_DIR / "rtsd-frames"

OUTPUT_DIR = (BASE_DIR / "data" / "mapping_data") if (BASE_DIR / "data").exists() else (BASE_DIR / "mapping_data")

ACTIVE_TRAIN_ANNO = ANNO_TRAIN_DEBUG if DEBUG else ANNO_TRAIN_FULL

if OUTPUT_DIR.exists():
    print(f" Удаление прошлой версии: {OUTPUT_DIR} ...")
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

 Удаление прошлой версии: /home/ruslana/Projects/RaspberryYolo/RaspberryRoadSign/data/mapping_data ...


In [11]:
if OUTPUT_DIR.exists():
    print(f" Очистка папки сборки:{OUTPUT_DIR} ...")
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

 Очистка папки сборки:/home/ruslana/Projects/RaspberryYolo/RaspberryRoadSign/data/mapping_data ...


List with classes

In [12]:
# Перевод строковых имен RTSD в последовательные индексы YOLO (0..102)
RTSD_NAME_TO_NEW_ID = {
    # WARNING
    '1_1': 82, '1_2': 55, '1_5': 88, '1_7': 76, '1_8': 37, '1_10': 100,
    '1_11': 42, '1_11_1': 25, '1_12': 45, '1_12_2': 43, '1_13': 71, '1_14': 72,
    '1_15': 18, '1_16': 24, '1_17': 2, '1_18': 78, '1_19': 22, '1_20': 44,
    '1_21': 69, '1_22': 34, '1_23': 1, '1_25': 11, '1_26': 98, '1_27': 35,
    '1_30': 92, '1_33': 16, 
    # PRIORITY
    '2_1': 0, '2_2': 8, '2_3': 39, '2_3_2': 36, '2_3_3': 40, '2_3_4': 73, 
    '2_4': 9, '2_5': 47, '2_6': 74, '2_7': 79,
    # PROHIBITORY
    '3_1': 48, '3_2': 50, '3_4': 12, '3_6': 97, '3_10': 60, '3_11': 84,
    '3_12': 94, '3_13': 38, '3_14': 54, '3_16': 91, '3_18': 28, '3_18_2': 75,
    '3_19': 77, '3_20': 49, '3_21': 70, '3_27': 17, '3_28': 63, '3_29': 89, 
    '3_30': 85, '3_31': 67, '3_32': 46, '3_33': 95,
    # Объединение номиналов скоростей в мета-класс 3
    '3_24': 3, '3_24_n20': 3, '3_24_n40': 3, '3_24_n60': 3, '3_24_n80': 3, 
    '3_24_n90': 3, '3_24_n100': 3, '3_24_n110': 3, '3_24_n120': 3, '3_25': 6, 
    # MANDATORY
    '4_1_1': 15, '4_1_2': 32, '4_1_3': 64, '4_1_4': 56, '4_1_5': 59, '4_1_6': 13, 
    '4_2_1': 10, '4_2_2': 61, '4_2_3': 14, '4_3': 58, '4_5': 83, 
    # OTHER / INFO
    '5_3': 66, '5_4': 65, '5_5': 30, '5_6': 29, '5_7_1': 86, '5_7_2': 87, 
    '5_8': 96, '5_11': 93, '5_12': 90, '5_14': 80, '5_16': 5, '5_17': 99, 
    '5_18': 52, '5_19_1': 4, '5_21': 81, '5_22': 51,
    # SERVICE
    '6_2': 68, '6_3_1': 19, '6_4': 23, '6_6': 26, '6_7': 20, '6_16': 7,
    # ADDITIONAL
    '7_1': 62, '7_2': 27, '7_3': 21, '7_4': 31, '7_5': 53, '7_6': 57,
    '7_7': 41, '7_11': 33, '7_14': 102, '7_18': 101
}

# Жестко зафиксированные проверенные текстовые имена (Очищенные от опечаток CSV)
NAMES_DICT_103 = {
    0: '2.1: Главная дорога', 1: '1.21: Дети', 2: '1.16.1: Искусственная неровность',
    3: '3.24: Ограничение максимальной скорости', 4: '5.16: Пешеходный переход',
    5: '5.12.1: Остановочный пункт автобуса', 6: '3.25: Конец зоны ограничения скорости',
    7: '5.33: Стоп-линия', 8: '2.2: Конец главной дороги', 9: '2.4: Уступить дорогу',
    10: '4.2.1: Объезд препятствия справа', 11: '1.23: Дорожные работы',
    12: '3.4: Движение грузовых ТС запрещено', 13: '4.1.6: Движение направо или налево',
    14: '4.2.3: Объезд препятствия справа или слева', 15: '4.1.1: Движение прямо',
    16: '1.30: Прочие опасности', 17: '3.27: Остановка запрещена', 18: '1.15: Скользкая дорога',
    19: '5.11.1: Место для разворота', 20: '5.17.3-4: Надземный пешеходный переход',
    21: '6.3: Автозаправочная станция', 22: '1.32: Опасная обочина', 23: '5.15: Место стоянки',
    24: '1.16.2-4: Неровная дорога', 25: '1.11.2: Опасный поворот (налево)',
    26: '5.17.1-2: Подземный пешеходный переход', 27: '6.2: Больница',
    28: '3.18.1: Поворот направо запрещён', 29: '5.6: Конец дороги с односторонним движением',
    30: '5.5: Дорога с односторонним движением', 31: '6.4: Техническое обслуживание автомобилей',
    32: '4.1.2: Движение направо', 33: '6.11: Место отдыха', 34: '1.20: Впереди пешеходный переход',
    35: '1.25: Дикие животные', 36: '2.3.2: Примыкание второстепенной дороги (справа)',
    37: '1.8: Светофорное регулирование', 38: '3.13: Ограничение высоты',
    39: '2.3.1: Пересечение со второстепенной дорогой', 40: '2.3.3: Примыкание второстепенной дороги (слева)',
    41: '6.7: Пункт питания', 42: '1.11.1: Опасный поворот (направо)',
    43: '1.12.2: Опасные повороты (первый — налево)', 44: '1.18.1: Сужение дороги с обеих сторон',
    45: '1.12.1: Опасные повороты (первый — направо)', 46: '3.32: Движение ТС с опасными грузами запрещено',
    47: '2.5: Движение без остановки запрещено', 48: '3.1: Въезд запрещён',
    49: '3.20: Обгон запрещён', 50: '3.2: Движение запрещено', 51: '5.39: Конец жилой зоны',
    52: '5.14.2: Место стоянки такси', 53: '6.5: Мойка автомобилей', 54: '3.14: Ограничение ширины',
    55: '1.2: Железнодорожный переезд без шлагбаума', 56: '4.1.4: Движение прямо или направо',
    57: '6.6: Телефон', 58: '4.3: Круговое движение', 59: '4.1.5: Движение прямо или налево',
    60: '3.10: Движение пешеходов запрещено', 61: '4.2.2: Объезд препятствия слева',
    62: '6.1: Пункт первой медицинской помощи', 63: '3.28: Стоянка запрещена', 64: '4.1.3: Движение налево',
    65: '5.4: Конец дороги для автомобилей', 66: '5.3: Дорога для автомобилей',
    67: '3.31: Конец зоны всех ограничений', 68: '5.18.1: Рекомендуемая скорость',
    69: '1.19: Двустороннее движение', 70: '3.21: Конец зоны запрещения обгона',
    71: '1.13: Крутой спуск', 72: '1.14: Крутой подъём', 73: '2.3.4: Пересечение равнозначных дорог',
    74: '2.6: Преимущество встречного движения', 75: '3.18.2: Поворот налево запрещён',
    76: '1.7: Пересечение с круговым движением', 77: '3.19: Разворот запрещён', 78: '1.17: Выброс щебня',
    79: '2.7: Преимущество перед встречным движением', 80: '5.9.1: Полоса для маршрутных ТС',
    81: '5.38: Жилая зона', 82: '1.1: Железнодорожный переезд со шлагбаумом', 83: '4.5.1: Пешеходная дорожка',
    84: '3.11: Ограничение массы', 85: '3.30: Стоянка запрещена по чётным числам',
    86: '5.7.1: Выезд на дорогу с односторонним движением', 87: '5.7.2: Выезд на дорогу с односторонним движением',
    88: '1.5: Пересечение с трамвайной линией', 89: '3.29: Стоянка запрещена по нечётным числам',
    90: '5.10.4: Конец дороги с полосой для маршрутных ТС', 91: '3.16: Ограничение минимальной дистанции',
    92: '1.28: Низколетящие самолеты', 93: '5.10.1: Дорога с полосой для маршрутных ТС',
    94: '3.12: Ограничение нагрузки на ось', 95: '3.33: Зона с ограничением максимальной скорости',
    96: '5.35: Реверсивное движение', 97: '3.6: Движение тракторов запрещено', 98: '1.24: Перегон скота',
    99: '5.13.1: Остановочный пункт трамвая', 100: '1.10: Выезд на набережную', 101: '6.13: Туалет',
    102: '6.14: Пункт контроля автомобильных перевозок'
}

In [13]:
from collections import defaultdict

def process_dataset_split(anno_json_path, images_src_dir, split_name):
    images_out = OUTPUT_DIR / split_name / "images"
    labels_out = OUTPUT_DIR / split_name / "labels"
    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)

    if not Path(anno_json_path).exists():
        print(f"⚠️ Файл {anno_json_path} отсутствует. Пропускаем шаг.")
        return

    with open(anno_json_path, "r") as f:
        anno_data = json.load(f)

    id_to_image = {img['id']: img for img in anno_data['images']}
    id_to_catname = {cat['id']: cat['name'] for cat in anno_data['categories']}

    img_to_annos = defaultdict(list)
    for a in anno_data['annotations']:
        img_to_annos[a['image_id']].append(a)

    copied_images = 0
    written_boxes = 0
    skipped_boxes = 0

    for img_id, img_info in tqdm(id_to_image.items(), desc=f"Сборка {split_name}"):
        filename = Path(img_info['file_name']).name
        img_src_path = Path(images_src_dir) / filename
        
        if not img_src_path.exists():
            continue

        img_w, img_h = img_info['width'], img_info['height']
        yolo_lines = []

        for a in img_to_annos[img_id]:
            cat_name = id_to_catname.get(a['category_id'], '')
            new_id = RTSD_NAME_TO_NEW_ID.get(cat_name)

            # Автоматическая фильтрация: если класса нет в словаре (RUSSIAN_ONLY) — пропускаем рамку
            if new_id is None:
                skipped_boxes += 1
                continue

            # Нормализация координат COCO в форматы YOLO
            x, y, w, h = a['bbox']
            xc = (x + w / 2) / img_w
            yc = (y + h / 2) / img_h
            wn = w / img_w
            hn = h / img_h

            xc, yc = max(0.0, min(1.0, xc)), max(0.0, min(1.0, yc))
            wn, hn = max(0.0, min(1.0, wn)), max(0.0, min(1.0, hn))
            
            yolo_lines.append(f"{new_id} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
            written_boxes += 1

        # Переносим кадр, только если на нем остался хотя бы один валидный знак
        if yolo_lines:
            shutil.copy2(img_src_path, images_out / filename)
            copied_images += 1
            
            label_name = Path(filename).stem + ".txt"
            with open(labels_out / label_name, "w") as lf:
                lf.write("\n".join(yolo_lines) + "\n")

    print(f"✓ Подвыборка {split_name} готова! Кадров: {copied_images}. Рамок: {written_boxes}. Отсечено РФ: {skipped_boxes}")

In [14]:
# 1. Запуск переработки
process_dataset_split(ACTIVE_TRAIN_ANNO, RTSD_FRAMES_DIR, "train")
process_dataset_split(ANNO_VAL, RTSD_FRAMES_DIR, "val")

# 2. Декларативное построение структуры манифеста
yaml_content = f"""train: {OUTPUT_DIR.resolve()}/train/images
val: {OUTPUT_DIR.resolve()}/val/images

nc: 103

names:
"""
for model_id in range(103):
    yaml_content += f"  - '{NAMES_DICT_103[model_id]}'\n"

# Запись конфигурационного файла в целевую директорию датасета
with open(OUTPUT_DIR / "data.yaml", "w", encoding="utf-8") as yf:
    yf.write(yaml_content)

print(f"\n🎉 ПАЙПЛАЙН СБОРКИ УСПЕШНО СРАБОТАТЬ!")
print(f"📁 Весь готовый датасет и манифест data.yaml лежат тут: {OUTPUT_DIR}")

Сборка train: 100%|██████████| 54188/54188 [00:26<00:00, 2082.10it/s]


✓ Подвыборка train готова! Кадров: 48901. Рамок: 78627. Отсечено РФ: 16865


Сборка val: 100%|██████████| 5000/5000 [00:02<00:00, 2222.62it/s]

✓ Подвыборка val готова! Кадров: 4516. Рамок: 7303. Отсечено РФ: 1563

🎉 ПАЙПЛАЙН СБОРКИ УСПЕШНО СРАБОТАТЬ!
📁 Весь готовый датасет и манифест data.yaml лежат тут: /home/ruslana/Projects/RaspberryYolo/RaspberryRoadSign/data/mapping_data
